# AS Betweenness Centrality, Weighted by Announced Address Space

This notebook downloads a BGP routing table (RIB) snapshot in MRT format from a
[RIPE RIS](https://ris.ripe.net/) route collector and computes the **betweenness
centrality of every transit AS**, where each AS path is weighted by the number of
IPv4 addresses of the prefix it leads to.

The metric follows Liu, Luo, Chang and Su,
["Characterizing Inter-domain Rerouting by Betweenness Centrality after Disruptive Events"](https://rockykcc.github.io/pub/JSAC-betweenness-centrality-13.pdf)
(IEEE JSAC, 2013), which defines AS betweenness centrality over the AS paths
actually observed in BGP rather than over shortest paths in a topology graph.
The data-handling approach (MRT parsing with `pybgpkit`, per-prefix address
counting with a `pytricia` longest-prefix-match trie) mirrors the
[nids-bgp-control-plane-key](https://github.com/CAIDA/nids-bgp-control-plane-key)
reference notebook.

## The metric

The paper (Eqn. 1) defines the betweenness centrality of an AS $v$ as

$$BC(v) = \frac{\sum_{u,w \in V} \sigma_{uw}(v)}{\sum_{u,w \in V} \sigma_{uw}}, \qquad u \neq w \neq v$$

where $\sigma_{uw}$ is the number of AS paths observed between $u$ and $w$, and
$\sigma_{uw}(v)$ is the number of those paths that pass **through** $v$ (as a
transit hop, not an endpoint). Because BGP is policy-routed, the paths are taken
directly from the collected routes, not recomputed as graph shortest paths.

**Adaptation to one RIB snapshot.** A RIB dump contains one best route per
*(peer AS, prefix)* pair. We treat each such route as one AS path from $u$ (the
collector's peer AS, the first hop) to $w$ (the origin AS, the last hop); every
AS strictly between them is a transit hop. With $P$ the set of all usable IPv4
routes in the RIB:

$$BC(v) = \frac{\left|\{\, p \in P : v \in \mathrm{transit}(p) \,\}\right|}{|P|}$$

**Address weighting.** The unweighted metric counts a path to a /8 and a path to
a /24 equally. To capture how much *address space* depends on an AS, we weight
each route by the size of its destination prefix:

$$BC_{\mathrm{w}}(v) = \frac{\sum_{p \in P,\ v \in \mathrm{transit}(p)} w\!\left(\mathrm{dst}(p)\right)}{\sum_{p \in P} w\!\left(\mathrm{dst}(p)\right)}$$

where $w(q)$ is the number of IPv4 addresses whose **longest matching prefix**
is $q$: the full size $2^{32-\mathrm{len}}$ of the prefix minus the addresses
covered by announced more-specifics inside it. This deduplication means every
routed address contributes its weight exactly once per vantage point, because
traffic to an address covered by a more-specific prefix follows the
more-specific route. Both scores lie in $[0, 1]$: $BC_{\mathrm{w}}(v)$ is the
fraction of routed address-space mass (address $\times$ vantage-point pairs)
whose control-plane route transits $v$.

## Setup

Install and import the required packages:
[`pybgpkit-parser`](https://github.com/bgpkit/bgpkit-parser) to parse MRT files,
[`pytricia`](https://github.com/jsommers/pytricia) for longest-prefix-match
lookups, and `pandas`/`matplotlib` for the results.

In [ ]:
%pip install -q pybgpkit-parser pytricia pandas matplotlib

import gc
import io
import time
import urllib.request
import ipaddress
from collections import defaultdict
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import pybgpkit_parser as bgpkit
import pytricia

## Data: a RIPE RIS `bview` snapshot

RIPE RIS collectors dump their full RIB every 8 hours (00:00, 08:00, 16:00 UTC)
as MRT `bview` files at
`https://data.ris.ripe.net/<collector>/<YYYY.MM>/bview.<YYYYMMDD>.<HHMM>.gz`.

The collector choice trades runtime for vantage points — more peers means more
$(u, w)$ path samples and a less biased centrality estimate:

| Collector | Location | File size (2026-08-01) |
|---|---|---|
| `rrc00` | Amsterdam (multihop, most peers) | ~414 MB |
| `rrc01` | London, LINX | ~347 MB |
| `rrc10` | Milan, MIX | ~129 MB |
| `rrc06` | Otemachi, JPNAP | ~42 MB |

`rrc00` is the default below; switch to `rrc06` for a quick test run
(the whole notebook then completes in a few minutes).

In [ ]:
COLLECTOR = "rrc00"
SNAPSHOT_DATE = "20260801"   # YYYYMMDD
SNAPSHOT_TIME = "0000"       # bviews exist at 0000, 0800, 1600 UTC

RIB_URL = (f"https://data.ris.ripe.net/{COLLECTOR}/"
           f"{SNAPSHOT_DATE[:4]}.{SNAPSHOT_DATE[4:6]}/"
           f"bview.{SNAPSHOT_DATE}.{SNAPSHOT_TIME}.gz")
RIB_PATH = Path(f"data/bview.{COLLECTOR}.{SNAPSHOT_DATE}.{SNAPSHOT_TIME}.gz")

def _report(count, block_size, total_size):
    done = count * block_size
    if done % (100 * 1024 * 1024) < block_size:
        print(f"  {done / 1e6:,.0f} MB...")

if RIB_PATH.exists():
    print(f"using cached {RIB_PATH} ({RIB_PATH.stat().st_size / 1e6:,.0f} MB)")
else:
    RIB_PATH.parent.mkdir(parents=True, exist_ok=True)
    print(f"downloading {RIB_URL}")
    t0 = time.time()
    urllib.request.urlretrieve(RIB_URL, RIB_PATH, reporthook=_report)
    print(f"done: {RIB_PATH.stat().st_size / 1e6:,.0f} MB in {time.time() - t0:.0f}s")

## Pass 1 — collect the announced IPv4 prefixes

The address weight of a prefix depends on which *other* prefixes are announced
inside it, so we need the complete set of announced prefixes before any path can
be weighted. This first pass over the MRT file only collects that set (IPv6
prefixes are excluded — address counts across the two families are not
comparable). The path-counting pass comes after the weights are built.

In [ ]:
t0 = time.time()
raw_prefixes = set()
n_entries = 0
for elem in bgpkit.Parser(url=str(RIB_PATH)):
    if elem.elem_type != "A":
        continue
    n_entries += 1
    pfx = elem.prefix
    if ":" not in pfx:  # keep IPv4 only
        raw_prefixes.add(pfx)
    if n_entries % 5000000 == 0:
        print(f"  {n_entries:,} entries, {len(raw_prefixes):,} IPv4 prefixes...")

print(f"{n_entries:,} RIB entries, {len(raw_prefixes):,} unique IPv4 prefixes "
      f"({time.time() - t0:.0f}s)")

## Address weight per prefix (longest-prefix-match deduplication)

A prefix of length $\ell$ spans $2^{32-\ell}$ addresses, but if a more-specific
prefix is announced inside it, traffic to those addresses follows the
more-specific route. Counting both at full size would double-count the nested
space. So, as in the reference notebook, each prefix is weighted by the
addresses for which it is the **longest match**: its full size minus the sizes
of the announced prefixes directly nested inside it. Subtracting only *direct*
children (prefixes whose immediate parent in the trie is this prefix) removes
each covered address exactly once, however deep the nesting goes.

Two consequences worth noting:

- A prefix completely covered by its more-specifics gets weight 0 — its paths
  still count toward the unweighted metric, but carry no address mass.
- A default route (`0.0.0.0/0`) would soak up all unannounced space, so it is
  dropped entirely (announcing a default to a collector is junk, not
  reachability).

The total of all weights should come out close to the routed IPv4 address space
(~3.1 billion addresses, ~72% of the 2³² total).

In [ ]:
t0 = time.time()
pyt = pytricia.PyTricia(32)
raw_to_norm = {}
for pfx in raw_prefixes:
    try:
        net = ipaddress.ip_network(pfx, strict=False)
    except ValueError:
        continue
    if net.prefixlen == 0:  # drop default routes
        continue
    norm = str(net)
    raw_to_norm[pfx] = norm
    pyt[norm] = True

norm_weight = {}
for pfx in pyt:
    w = 1 << (32 - int(pfx.split("/")[1]))
    for child in pyt.children(pfx):
        if child != pfx and pyt.parent(child) == pfx:
            w -= 1 << (32 - int(child.split("/")[1]))
    norm_weight[pfx] = w

# keyed by the raw prefix strings seen in the MRT file, for direct lookup in pass 2
weights = {raw: norm_weight[norm] for raw, norm in raw_to_norm.items()}

total_routed = sum(norm_weight.values())
zero_w = sum(1 for w in norm_weight.values() if w == 0)
print(f"{len(norm_weight):,} prefixes weighted ({time.time() - t0:.0f}s)")
print(f"  total routed IPv4 space: {total_routed:,} addresses "
      f"({100 * total_routed / 2**32:.1f}% of 2^32)")
print(f"  fully covered by more-specifics (weight 0): {zero_w:,} prefixes")

## Pass 2 — accumulate transit counts per AS

Now iterate over every route again and credit each transit AS on its AS path.
Per route:

- **AS-path prepending** (the same AS repeated consecutively to make a path less
  attractive) is collapsed — it is a traffic-engineering artifact, not extra hops.
- Paths containing an **AS-set** (`{...}`, from route aggregation) are skipped;
  the actual sequence of ASes is ambiguous. These are rare (well under 0.1%).
- The first hop (the collector's peer AS, $u$) and the last hop (the origin AS,
  $w$) are endpoints, not transit — only the ASes strictly between them are
  credited, and at most once per path even if a path loops.
- Every usable route counts in the denominator, including peer-to-origin routes
  with no transit hop at all.

**Performance note:** the lookup structures built above hold millions of
long-lived Python objects. The allocations inside this loop keep triggering
full garbage collections that re-traverse all of them, slowing the loop by
~80×. `gc.freeze()` moves the existing heap out of the collector's scope
(measured on `rrc06`: 2848s → 33s).

In [ ]:
gc.collect()
gc.freeze()

t0 = time.time()
transit_w = defaultdict(int)   # AS -> address-weighted path mass
transit_u = defaultdict(int)   # AS -> path count
total_weight = 0
total_paths = 0
n_asset = 0
n_seen = 0
for elem in bgpkit.Parser(url=str(RIB_PATH)):
    if elem.elem_type != "A":
        continue
    n_seen += 1
    if n_seen % 5000000 == 0:
        print(f"  {n_seen:,} entries, {total_paths:,} paths counted...")
    w = weights.get(elem.prefix)
    if w is None:  # IPv6, default route, or unparseable prefix
        continue
    path_str = elem.as_path
    if path_str is None:
        continue
    if "{" in path_str:  # AS-set: ambiguous, skip
        n_asset += 1
        continue
    path = []
    prev = None
    for hop in path_str.split():
        if hop != prev:  # collapse prepending
            path.append(hop)
            prev = hop
    total_paths += 1
    total_weight += w
    for asn in set(path[1:-1]):
        transit_u[asn] += 1
        transit_w[asn] += w

gc.unfreeze()
print(f"{total_paths:,} IPv4 paths counted ({n_asset:,} skipped for AS-sets), "
      f"{len(transit_u):,} transit ASes ({time.time() - t0:.0f}s)")

## Normalize and rank

Divide by the totals to get $BC(v)$ and $BC_{\mathrm{w}}(v)$, attach AS names
and countries from RIPE's [asn.txt](https://ftp.ripe.net/ripe/asnames/asn.txt),
and rank. The `paths` and `addresses` columns are the raw numerators: how many
routes transit the AS, and how much address $\times$ vantage mass they carry.

In [ ]:
ASNAMES_URL = "https://ftp.ripe.net/ripe/asnames/asn.txt"
asn_info = {}
try:
    with urllib.request.urlopen(ASNAMES_URL, timeout=60) as resp:
        for line in io.TextIOWrapper(resp, encoding="utf-8", errors="replace"):
            num, _, rest = line.strip().partition(" ")
            if not num.isdigit():
                continue
            name, sep, country = rest.rpartition(", ")
            asn_info[int(num)] = (name if sep else rest, country if sep else "")
except Exception as exc:
    print(f"could not fetch AS names ({exc}); continuing with numbers only")

rows = []
for asn_s, wmass in transit_w.items():
    asn = int(asn_s)
    name, country = asn_info.get(asn, ("", ""))
    rows.append({
        "asn": asn,
        "name": name[:48],
        "country": country,
        "bc_weighted": wmass / total_weight,
        "bc_unweighted": transit_u[asn_s] / total_paths,
        "paths": transit_u[asn_s],
        "addresses": wmass,
    })

df = (pd.DataFrame(rows)
        .sort_values("bc_weighted", ascending=False)
        .reset_index(drop=True))
df.insert(0, "rank_w", df.index + 1)
df["rank_u"] = df["bc_unweighted"].rank(ascending=False, method="min").astype(int)

df.head(25).style.format({
    "bc_weighted": "{:.4f}",
    "bc_unweighted": "{:.4f}",
    "paths": "{:,}",
    "addresses": "{:,}",
}).hide(axis="index")

## Distribution of centrality across transit ASes

The CCDF below shows, for each centrality value $x$, how many transit ASes have
$BC \geq x$. Like the prefix- and address-count distributions in the reference
notebook, expect a heavy tail on log-log axes: most transit ASes carry a tiny
fraction of paths, while a handful of tier-1 and large tier-2 networks each
transit a sizable share of the routed address space.

In [ ]:
INK, INK2, MUTED = "#0b0b0b", "#52514e", "#898781"
GRID, SURFACE = "#e1e0d9", "#fcfcfb"
BLUE, ORANGE = "#2a78d6", "#eb6834"

def style_axes(ax):
    ax.set_facecolor(SURFACE)
    for spine in ax.spines.values():
        spine.set_color(MUTED)
        spine.set_linewidth(0.8)
    ax.tick_params(colors=MUTED, labelcolor=INK2)
    ax.grid(True, which="both", color=GRID, linewidth=0.6, alpha=0.6)
    ax.set_axisbelow(True)

def ccdf(values):
    xs = sorted(values)
    return xs, [len(xs) - i for i in range(len(xs))]

fig, ax = plt.subplots(figsize=(7, 4.5))
fig.patch.set_facecolor(SURFACE)
style_axes(ax)

xw, yw = ccdf(df["bc_weighted"])
xu, yu = ccdf(df["bc_unweighted"])
ax.plot(xw, yw, color=BLUE, linewidth=2, label="address-weighted")
ax.plot(xu, yu, color=ORANGE, linewidth=2, label="unweighted (path count)")

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("AS betweenness centrality", color=INK2)
ax.set_ylabel("Transit ASes with BC \u2265 x", color=INK2)
ax.set_title(f"CCDF of AS betweenness centrality ({COLLECTOR}, {SNAPSHOT_DATE})",
             color=INK)
ax.legend(frameon=False, labelcolor=INK2)
fig.tight_layout()
plt.show()

## What the address weighting changes

Each point below is one transit AS. On the diagonal, weighting does not matter:
the AS transits an address-typical mix of prefixes. Above the diagonal, the AS
carries routes to *larger-than-average* prefixes (e.g. carriers in front of
legacy /8s and big cloud or telco aggregates) — the weighted metric promotes
it. Below the diagonal, its paths mostly lead to small prefixes (long /24-heavy
deaggregation), so its path count overstates the address space that depends on
it. The furthest-off-diagonal ASes are where this metric tells a different
story than simple path counting.

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 6))
fig.patch.set_facecolor(SURFACE)
style_axes(ax)

lo = max(df["bc_unweighted"].min(), df["bc_weighted"].min(), 1e-8)
hi = max(df["bc_unweighted"].max(), df["bc_weighted"].max()) * 2
ax.plot([lo, hi], [lo, hi], color=MUTED, linewidth=1, linestyle="--", zorder=1)
ax.annotate("equal under both metrics", xy=(hi, hi),
            xytext=(-8, -14), textcoords="offset points",
            ha="right", fontsize=8, color=MUTED)

ax.scatter(df["bc_unweighted"], df["bc_weighted"],
           s=14, color=BLUE, alpha=0.45, linewidths=0, zorder=2)

for _, row in df.head(8).iterrows():
    ax.annotate(f"AS{row['asn']}", xy=(row["bc_unweighted"], row["bc_weighted"]),
                xytext=(5, 3), textcoords="offset points",
                fontsize=8, color=INK2)

ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlim(lo, hi)
ax.set_ylim(lo, hi)
ax.set_xlabel("Unweighted betweenness centrality (path count)", color=INK2)
ax.set_ylabel("Address-weighted betweenness centrality", color=INK2)
ax.set_title(f"Address weighting vs. path counting ({COLLECTOR}, {SNAPSHOT_DATE})",
             color=INK)
fig.tight_layout()
plt.show()

## Interpretation and caveats

- **Vantage-point bias.** The paths are those visible from this collector's
  peers, so ASes topologically close to the peers are over-represented. A run
  on `rrc06` (Tokyo), for example, ranks Japanese carriers (KDDI AS2516,
  IIJ AS2497, ARTERIA AS2518) far higher than a global view would. `rrc00` has
  the most diverse peer set; merging several collectors reduces the bias
  further.
- **Peers are endpoints.** A collector peer appears as the first hop $u$ of its
  own routes and never earns transit credit from them — large ASes that peer
  with the collector are structurally *under*-counted.
- **Control plane, not traffic.** A path weighted by addresses says how much
  address space *would* traverse the AS if every address mattered equally; real
  traffic per address varies over orders of magnitude.
- **One snapshot.** The paper's actual object of study is the *variance*
  $\widetilde{BC}_t(v) = BC_t(v) - BC_{t-1}(v)$ across a sequence of time slots,
  which localizes disruptive events. Re-running this notebook over consecutive
  `bview`/update intervals and differencing the per-AS scores is the natural
  extension.
- **No bogon filtering.** Announcements of reserved or unallocated space are
  counted like any other; a stricter version would drop them before weighting.